In [1]:
import numpy as np

def update_cg(var, r, q, Aq, rold):
    alpha = rold / np.inner(q, Aq)
    var = var + alpha * q
    r = r - alpha * Aq
    rnew = np.inner(r, r)
    q = r + (rnew / rold) * q
    return var, r, q, rnew

def compute_mape(var, var_hat):
    return np.sum(np.abs(var - var_hat) / var) / var.shape[0]

def compute_rmse(var, var_hat):
    return np.sqrt(np.sum((var - var_hat) ** 2) / var.shape[0])

def forecast(series, mu, W, V, d, horizon):
    N, t = series.shape
    R = W.shape[1]
    est = np.zeros((N, horizon))
    for h in range(horizon):
        Xt = np.zeros(d * N)
        for k in range(d):
            Xt[k * N : (k + 1) * N] = series[:, t - horizon + h - (k + 1)]
        est[:, h] = mu + W @ (V @ Xt)
    return est

In [3]:
def solve_mu(Y, p, mu, W, V, delta, d, R, max_iter = 50):
    N, T = Y.shape
    if N * d * (T - d) <= 1e+8:
        X = np.empty((N * d, T - d))
        for k in range(d):
            X[k * N : (k + 1) * N, :] = Y[:, d - k - 1 : T - k - 1]
        C = Y[:, d :] - W @ V @ X
    else:
        temp = np.empty((R, T - d))
        for t in range(d, T):
            temp[:, t - d] = V @ np.hstack([Y[:, t - k - 1] for k in range(d)])
        C = Y[:, d :] - W @ temp
    for iter in range(max_iter):
    # for iter in range(1):
        epsilon = Y[:, d :] - mu[:, np.newaxis] - C
        theta = np.ones((N, T - d))
        pos = np.where(np.abs(epsilon) > delta)
        theta[pos] = delta / np.abs(epsilon[pos])
        ######### p = 0 is the log-Huber ##########
        theta[pos] = delta ** (2 - p) / (np.abs(epsilon[pos]) ** (2 - p))
        ###########################################

        mu = np.einsum('nt, nt -> n', theta, C, optimize = 'greedy') / np.sum(theta, axis = 1)
    return mu

In [4]:
def ell_W(temp, W):
    return np.einsum('nwk, nw -> nk', temp, W, optimize = 'greedy').reshape(-1, order = 'F')

def solve_W(Y, p, mu, W, V, delta, d, R, max_iter = 50):
    N, T = Y.shape
    if N * d * (T - d) <= 1e+8:
        X = np.empty((N * d, T - d))
        for k in range(d):
            X[k * N : (k + 1) * N, :] = Y[:, d - k - 1 : T - k - 1]
        var1 = V @ X
    else:
        var1 = np.empty((R, T - d))
        for t in range(d, T):
            var1[:, t - d] = V @ np.hstack([Y[:, t - k - 1] for k in range(d)])
    temp1 = np.einsum('wt, kt -> wkt', var1, var1, optimize = 'greedy')
    data = Y[:, d :] - mu[:, np.newaxis]
    for iter in range(max_iter):
    # for iter in range(1):
        epsilon = data - W @ var1
        theta = np.ones((N, T - d))
        pos = np.where(np.abs(epsilon) > delta)
        theta[pos] = delta / np.abs(epsilon[pos])
        ######### p = 0 is the log-Huber ##########
        theta[pos] = delta ** (2 - p) / (np.abs(epsilon[pos]) ** (2 - p))
        ###########################################

        w = np.reshape(W, -1, order = 'F')
        response = np.einsum('nt, wt -> nw', theta * data, var1, optimize = 'greedy').reshape(-1, order = 'F')
        temp = np.einsum('nt, wkt -> nwk', theta, temp1, optimize = 'greedy')
        r = response - ell_W(temp, W)
        q = r.copy()
        rold = np.inner(r, r)
        for it in range(5): # CG iteration
            Q = np.reshape(q, (N, R), order = 'F')
            Aq = ell_W(temp, Q)
            w, r, q, rold = update_cg(w, r, q, Aq, rold)
        W = np.reshape(w, (N, R), order = 'F')
    return W

In [5]:
def ell_V(temp, X, V):
    return np.einsum('kht, ht, lt -> kl', temp, V @ X, X, optimize = 'greedy').reshape(-1, order = 'F')

def solve_V(Y, p, mu, W, V, delta, d, R, max_iter = 50):
    N, T = Y.shape
    X = np.zeros((N * d, T - d))
    for k in range(d):
        X[k * N : (k + 1) * N, :] = Y[:, d - k - 1 : T - k - 1]
    data = Y[:, d :] - mu[:, np.newaxis]
    for iter in range(max_iter):
    # for iter in range(1):
        epsilon = data - W @ (V @ X)
        theta = np.ones((N, T - d))
        pos = np.where(np.abs(epsilon) > delta)
        theta[pos] = delta / np.abs(epsilon[pos])
        ######### p = 0 is the log-Huber ##########
        theta[pos] = delta ** (2 - p) / (np.abs(epsilon[pos]) ** (2 - p))
        ###########################################

        v = np.reshape(V, -1, order = 'F')
        response = (W.T @ (theta * data) @ X.T).reshape(-1, order = 'F')
        temp = np.einsum('nt, nk, nh -> kht', theta, W, W, optimize = 'greedy')
        r = response - ell_V(temp, X, V)
        q = r.copy()
        rold = np.inner(r, r)
        for it in range(5): # CG iteration
            Q = np.reshape(q, (R, d * N), order = 'F')
            Aq = ell_V(temp, X, Q)
            v, r, q, rold = update_cg(v, r, q, Aq, rold)
        V = np.reshape(v, (R, d * N), order = 'F')
    return V

In [7]:
import numpy as np
import time

def ell_v_initial(var, X, V):
    return (var @ (V @ X) @ X.T).reshape(-1, order = 'F')

def initialize_WV(Y, d, R):
    N, T = Y.shape
    X = np.zeros((N * d, T - d))
    for k in range(d):
        X[k * N : (k + 1) * N, :] = Y[:, d - k - 1 : T - k - 1]
    mat, _, _ = np.linalg.svd(Y[:, d :], full_matrices = False)
    W = mat[:, : R]
    V = np.zeros((R, d * N))
    v = np.reshape(V, -1, order = 'F')
    response = np.einsum('nr, nt, kt -> rk', W, Y[:, d :], X, optimize = 'greedy').reshape(-1, order = 'F')
    var = W.T @ W
    r = response - ell_v_initial(var, X, V)
    q = r.copy()
    rold = np.inner(r, r)
    for it in range(5): # CG iteration
        Q = np.reshape(q, (R, d * N), order = 'F')
        Aq = ell_v_initial(var, X, Q)
        v, r, q, rold = update_cg(v, r, q, Aq, rold)
    V = np.reshape(v, (R, d * N), order = 'F')
    return W, V

def huber_var_mf(Y, p, delta, d, R, max_iter = 50):
    N = Y.shape[0]
    mu = np.zeros(N)
    W, V = initialize_WV(Y, d, R)
    for iter in range(max_iter):
        mu = solve_mu(Y, p, mu, W, V, delta, d, R)
        W = solve_W(Y, p, mu, W, V, delta, d, R)
        V = solve_V(Y, p, mu, W, V, delta, d, R)
    return mu, W, V

In [9]:
import time
import pandas as pd
import cupy as np

data = pd.read_csv('top_pages.csv.gz', compression='gzip').iloc[:, 2 :].values

for p in [1, 2/3, 1/2, 1/3, 0]:
    print('p = {}'.format(p))
    horizon = 168
    d = 7 * 24
    for R in [3]:
        for delta in [40]:
            print('Rank = {}'.format(R))
            print('Delta = {}'.format(delta))
            start = time.time()
            np.random.seed(0)
            mu, W, V = huber_var_mf(np.array(data[:, : 17 * 24]), p, delta, d, R)
            end = time.time()
            print('Running time: %d seconds'%(end - start))

            ground = np.array(data[:, 24 * 17 : 24 * 24])
            est = forecast(np.array(data[:, : 24 * 24]), mu, W, V, d, horizon)
            pos = np.where(ground > 0)
            print('MAPE: {:.6}'.format(compute_mape(ground[pos], est[pos])))
            print('RMSE: {:.6}'.format(compute_rmse(ground[pos], est[pos])))
            print()

    ground = np.array(data[:, - horizon :])
    est = forecast(np.array(data), mu, W, V, d, horizon)
    pos = np.where(ground > 0)
    print('MAPE: {:.6}'.format(compute_mape(ground[pos], est[pos])))
    print('RMSE: {:.6}'.format(compute_rmse(ground[pos], est[pos])))
    print()

p = 1
Rank = 3
Delta = 40
Running time: 499 seconds
MAPE: 2.06442
RMSE: 1181.22

MAPE: 4.10409
RMSE: 1559.75

p = 0.6666666666666666
Rank = 3
Delta = 40
Running time: 499 seconds
MAPE: 2.171
RMSE: 1201.84

MAPE: 4.31594
RMSE: 1600.2

p = 0.5
Rank = 3
Delta = 40
Running time: 500 seconds
MAPE: 2.22699
RMSE: 1344.49

MAPE: 4.44811
RMSE: 1659.04

p = 0.3333333333333333
Rank = 3
Delta = 40
Running time: 499 seconds
MAPE: 2.22318
RMSE: 1390.75

MAPE: 4.4195
RMSE: 1738.7

p = 0
Rank = 3
Delta = 40
Running time: 499 seconds
MAPE: 2.18832
RMSE: 1317.53

MAPE: 4.30626
RMSE: 1671.88



In [8]:
import time
import pandas as pd
import cupy as np

data = pd.read_csv('top_pages.csv.gz', compression='gzip').iloc[:, 2 :].values

for p in [1, 2/3, 1/2, 1/3, 0]:
    print('p = {}'.format(p))
    horizon = 168
    d = 7 * 24
    for R in [4]:
        for delta in [40]:
            print('Rank = {}'.format(R))
            print('Delta = {}'.format(delta))
            start = time.time()
            np.random.seed(0)
            mu, W, V = huber_var_mf(np.array(data[:, : 17 * 24]), p, delta, d, R)
            end = time.time()
            print('Running time: %d seconds'%(end - start))

            ground = np.array(data[:, 24 * 17 : 24 * 24])
            est = forecast(np.array(data[:, : 24 * 24]), mu, W, V, d, horizon)
            pos = np.where(ground > 0)
            print('MAPE: {:.6}'.format(compute_mape(ground[pos], est[pos])))
            print('RMSE: {:.6}'.format(compute_rmse(ground[pos], est[pos])))
            print()

    ground = np.array(data[:, - horizon :])
    est = forecast(np.array(data), mu, W, V, d, horizon)
    pos = np.where(ground > 0)
    print('MAPE: {:.6}'.format(compute_mape(ground[pos], est[pos])))
    print('RMSE: {:.6}'.format(compute_rmse(ground[pos], est[pos])))
    print()

p = 1
Rank = 4
Delta = 40
Running time: 508 seconds
MAPE: 1.96651
RMSE: 1140.64

MAPE: 3.9731
RMSE: 1534.7

p = 0.6666666666666666
Rank = 4
Delta = 40
Running time: 503 seconds
MAPE: 2.04957
RMSE: 1137.41

MAPE: 4.15762
RMSE: 1553.07

p = 0.5
Rank = 4
Delta = 40
Running time: 503 seconds
MAPE: 2.07357
RMSE: 1173.6

MAPE: 4.17814
RMSE: 1617.02

p = 0.3333333333333333
Rank = 4
Delta = 40
Running time: 503 seconds
MAPE: 2.09593
RMSE: 1307.51

MAPE: 4.27303
RMSE: 1602.73

p = 0
Rank = 4
Delta = 40
Running time: 503 seconds
MAPE: 2.02174
RMSE: 1153.75

MAPE: 4.08144
RMSE: 1550.52



In [13]:
import numpy as np

data = np.load('daymet_tmax_na_2010.npz')['arr_0']
for t in [11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]:
    data = np.append(data, np.load('daymet_tmax_na_20{}.npz'.format(t))['arr_0'], axis = 1)

In [14]:
import time
import cupy as np

for p in [1, 2/3, 1/2, 1/3, 0]:
    print('p = {}'.format(p))
    horizon = 365
    d = 30
    for R in [6]:
        for delta in [1]:
            print('Rank = {}'.format(R))
            print('Delta = {}'.format(delta))
            start = time.time()
            np.random.seed(0)
            mu, W, V = huber_var_mf(np.array(data[:, : - (366 + 365)]), p, delta, d, R)
            end = time.time()
            print('Running time: %d seconds'%(end - start))

            ground = np.array(data[:, - (366 + 365) : - 365])
            est = forecast(np.array(data[:, : - 365]), mu, W, V, d, horizon)
            pos = np.where(ground != 0)
            print('MAPE: {:.6}'.format(compute_mape(ground[pos], est[pos])))
            print('RMSE: {:.6}'.format(compute_rmse(ground[pos], est[pos])))
            print()

    ground = np.array(data[:, - horizon :])
    est = forecast(np.array(data), mu, W, V, d, horizon)
    pos = np.where(ground != 0)
    print('MAPE: {:.6}'.format(compute_mape(ground[pos], est[pos])))
    print('RMSE: {:.6}'.format(compute_rmse(ground[pos], est[pos])))
    print()

p = 1
Rank = 6
Delta = 1
Running time: 751 seconds
MAPE: 0.183038
RMSE: 3.88579

MAPE: 0.36279
RMSE: 3.6013

p = 0.6666666666666666
Rank = 6
Delta = 1
Running time: 751 seconds
MAPE: 0.181256
RMSE: 3.91252

MAPE: 0.372995
RMSE: 3.63612

p = 0.5
Rank = 6
Delta = 1
Running time: 750 seconds
MAPE: 0.180838
RMSE: 3.92704

MAPE: 0.366708
RMSE: 3.65732

p = 0.3333333333333333
Rank = 6
Delta = 1
Running time: 751 seconds
MAPE: 0.180262
RMSE: 3.94231

MAPE: 0.36955
RMSE: 3.68008

p = 0
Rank = 6
Delta = 1
Running time: 751 seconds
MAPE: 0.17832
RMSE: 3.96199

MAPE: 0.354464
RMSE: 3.7215



In [15]:
import time
import cupy as np

for p in [1, 2/3, 1/2, 1/3, 0]:
    print('p = {}'.format(p))
    horizon = 365
    d = 30
    for R in [5]:
        for delta in [1]:
            print('Rank = {}'.format(R))
            print('Delta = {}'.format(delta))
            start = time.time()
            np.random.seed(0)
            mu, W, V = huber_var_mf(np.array(data[:, : - (366 + 365)]), p, delta, d, R)
            end = time.time()
            print('Running time: %d seconds'%(end - start))

            ground = np.array(data[:, - (366 + 365) : - 365])
            est = forecast(np.array(data[:, : - 365]), mu, W, V, d, horizon)
            pos = np.where(ground != 0)
            print('MAPE: {:.6}'.format(compute_mape(ground[pos], est[pos])))
            print('RMSE: {:.6}'.format(compute_rmse(ground[pos], est[pos])))
            print()

    ground = np.array(data[:, - horizon :])
    est = forecast(np.array(data), mu, W, V, d, horizon)
    pos = np.where(ground != 0)
    print('MAPE: {:.6}'.format(compute_mape(ground[pos], est[pos])))
    print('RMSE: {:.6}'.format(compute_rmse(ground[pos], est[pos])))
    print()

p = 1
Rank = 5
Delta = 1
Running time: 751 seconds
MAPE: 0.186083
RMSE: 4.02082

MAPE: 0.463237
RMSE: 3.7222

p = 0.6666666666666666
Rank = 5
Delta = 1
Running time: 750 seconds
MAPE: 0.185369
RMSE: 4.03105

MAPE: 0.515894
RMSE: 3.74762

p = 0.5
Rank = 5
Delta = 1
Running time: 750 seconds
MAPE: 0.185093
RMSE: 4.0374

MAPE: 0.528395
RMSE: 3.76101

p = 0.3333333333333333
Rank = 5
Delta = 1
Running time: 749 seconds
MAPE: 0.185308
RMSE: 4.04341

MAPE: 0.545006
RMSE: 3.77858

p = 0
Rank = 5
Delta = 1
Running time: 750 seconds
MAPE: 0.185138
RMSE: 4.05907

MAPE: 0.556322
RMSE: 3.80753

